# Day 2 · 02 · Performance, Lakeflow Jobs and DABs for BI Products

![Lakeflow Job DAG](../../assets/images/lakeflow_job_dag.png)

This notebook turns the Databricks + Power BI dataset into an operational BI
product: performance checks, cost guardrails, refresh strategy, Lakeflow Jobs
and DABs orientation.

## Learning objectives

By the end of this notebook you can:

- compare detail and aggregate query patterns,
- read basic performance symptoms,
- define BI cost guardrails,
- design a Lakeflow Job DAG,
- explain why DABs make deployment repeatable,
- complete a report certification checklist.

In [ ]:
%run ../../setup/00_setup

In [ ]:
required_tables = [
    f"{GOLD}.fact_sales_dashboard",
    f"{GOLD}.fact_sales_dashboard_monthly",
    f"{GOLD}.v_fact_sales_incremental",
    f"{GOLD}.kpi_monthly",
]

missing = [t for t in required_tables if not spark.catalog.tableExists(t)]
if missing:
    print("[BLOCKED] Missing required tables:")
    for t in missing:
        print("  -", t)
    print()
    print("Run this notebook first: notebooks/demo/day2_01_powerbi_semantic_model.ipynb")
    raise Exception("Pre-check failed: missing prerequisite tables. Run \"notebooks/demo/day2_01_powerbi_semantic_model.ipynb\" first.")

print("[OK] Pre-check passed, all required tables exist:")
for t in required_tables:
    print("  -", t)

## Business scenario

The dashboard now has trusted data, but production concerns remain:

- some report pages are slow,
- DirectQuery can multiply SQL Warehouse usage,
- refresh must be repeatable,
- the job definition should not live only in the UI.

## 1. Performance before/after

![Query profile reading map](../../assets/images/query_profile_reading_map.png)

Compare detail-grain and aggregate-grain queries for the same business
question.

In [ ]:
spark.sql(f"""
EXPLAIN FORMATTED
SELECT category, ROUND(SUM(line_revenue), 2) AS revenue
FROM {GOLD}.fact_sales_dashboard
WHERE is_completed
GROUP BY category
ORDER BY revenue DESC
""").show(truncate=False)

In [ ]:
spark.sql(f"""
EXPLAIN FORMATTED
SELECT category, ROUND(SUM(revenue), 2) AS revenue
FROM {GOLD}.fact_sales_dashboard_monthly
GROUP BY category
ORDER BY revenue DESC
""").show(truncate=False)

Discussion:

- Which query scans more rows?
- Which one should back a summary page?
- Which one is acceptable for a drill-through page?
- What happens if this is DirectQuery and every visual runs it repeatedly?

## 2. Query Profile walkthrough

`[UI DEMO]` Run one query from detail and one from aggregate, then open Query
Profile / Query History.

Look for:

- duration,
- rows scanned,
- bytes scanned,
- joins,
- shuffle/exchange,
- filters pushed down,
- user and warehouse.

## 3. Cost guardrails

Use `docs/templates/cost-awareness-checklist.md`.

| Area | Guardrail |
|---|---|
| Warehouse size | start small, scale only with evidence |
| Auto-stop | 5-10 minutes for training/demo |
| Import mode | default for executive dashboard |
| DirectQuery | exception, only on Gold objects |
| Aggregates | summary visuals use aggregate tables |
| Monitoring | review Query History and billing/system tables |

## 4. Refresh strategy

Use `docs/templates/refresh-strategy.md`.

Recommended BI refresh flow:

1. validate Silver sources,
2. refresh Gold dimensions and facts,
3. build KPI and BI aggregates,
4. validate reconciliation,
5. refresh Power BI dataset or expose updated Gold objects,
6. record certification status.

## 5. Lakeflow Jobs orientation

Adapted from `04_orchestration_demo.ipynb`.

| Task | Purpose | Failure behavior |
|---|---|---|
| `validate_sources` | confirm required Silver tables and bad-data expectations | fail fast |
| `refresh_gold` | build dimensions, fact and dashboard table | retry once |
| `build_bi_objects` | build KPI, aggregates and incremental view | retry once |
| `validate_bi_readiness` | reconciliation and certification checks | fail and notify owner |

`[UI DEMO]` Show Jobs UI: task graph, run details, logs, repair run and
parameters.

## 6. DABs / Declarative Automation Bundles

![DABs deployment flow](../../assets/images/dabs_deployment_flow.png)

DABs answers a simple operational question: can we recreate the job in another
environment from code instead of clicking through the UI again?

The reference file is `bundle/databricks.yml`.

In [ ]:
# Reference-only: inspect the generated bundle file from the repo.
# In a Databricks workspace repo, this path may differ. Treat this cell as
# documentation when running outside the local repo context.
print("Reference DABs file: bundle/databricks.yml")
print("CLI commands to run outside this notebook:")
print("  databricks bundle validate -t dev")
print("  databricks bundle deploy -t dev")
print("  databricks bundle run refresh_gold_bi_dataset -t dev")

## 7. Report certification checklist

Use `docs/templates/report-certification-checklist.md`.

| Check | Required evidence |
|---|---|
| KPI approved | KPI dictionary filled |
| Grain known | fact and aggregate grain documented |
| Source object chosen | BI contract filled |
| Refresh defined | refresh strategy filled |
| Cost guardrails defined | checklist filled |
| Reconciliation passed | max diff within tolerance |
| Owner named | business and technical owner |

In [ ]:
recon = spark.sql(f"""
WITH agg AS (
  SELECT year_month, ROUND(SUM(revenue), 2) AS revenue
  FROM {GOLD}.fact_sales_dashboard_monthly
  GROUP BY year_month
),
kpi AS (
  SELECT year_month, ROUND(revenue, 2) AS revenue
  FROM {GOLD}.kpi_monthly
)
SELECT MAX(ABS(agg.revenue - kpi.revenue)) AS max_diff
FROM agg JOIN kpi ON agg.year_month = kpi.year_month
""").first()["max_diff"]

assert recon < 0.01, f"Certification failed: revenue reconciliation diff {recon}"
print("[OK] Certification reconciliation passed")

## 8. Capstone

Final package participants should be able to explain:

- Gold KPI package,
- Power BI semantic dataset,
- Import vs DirectQuery decision,
- performance risk and mitigation,
- Lakeflow Job DAG,
- DABs deployment story,
- certification checklist.

This is the handoff from a notebook exercise to a real BI product pattern.

## Summary

After this notebook, the training has covered the full loop:

1. explore data and warehouse cost,
2. build Gold,
3. define KPI,
4. prepare Power BI,
5. validate performance,
6. automate refresh,
7. document and certify the report.